You can download the `requirements.txt` for this course from the workspace of this lab. `File --> Open...`

# L2: Create Agents to Research and Write an Article

In this lesson, you will be introduced to the foundational concepts of multi-agent systems and get an overview of the crewAI framework.

The libraries are already installed in the classroom. If you're running this notebook on your own machine, you can install the following:
```Python
!pip install crewai==0.28.8 crewai_tools==0.1.6 langchain_community==0.0.29
```

In [1]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

- Import from the crewAI libray.

In [2]:
from crewai import Agent, Task, Crew

- As a LLM for your agents, you'll be using OpenAI's `gpt-3.5-turbo`.

**Optional Note:** crewAI also allow other popular models to be used as a LLM for your Agents. You can see some of the examples at the [bottom of the notebook](#1).

In [3]:
import os
from utils import get_openai_api_key

openai_api_key = get_openai_api_key()
os.environ["OPENAI_MODEL_NAME"] = 'gpt-3.5-turbo'

## Creating Agents

- Define your Agents, and provide them a `role`, `goal` and `backstory`.
- It has been seen that LLMs perform better when they are role playing.

### Agent: Planner

**Note**: The benefit of using _multiple strings_ :
```Python
varname = "line 1 of text"
          "line 2 of text"
```

versus the _triple quote docstring_:
```Python
varname = """line 1 of text
             line 2 of text
          """
```
is that it can avoid adding those whitespaces and newline characters, making it better formatted to be passed to the LLM.

In [4]:
planner = Agent(
    role="Content Planner",
    goal="Plan engaging and factually accurate content on {topic}",
    backstory="You're working on planning a blog article "
              "about the topic: {topic}."
              "You collect information that helps the "
              "audience learn something "
              "and make informed decisions. "
              "Your work is the basis for "
              "the Content Writer to write an article on this topic.",
    allow_delegation=False,
	verbose=True
)

### Agent: Writer

In [5]:
writer = Agent(
    role="Content Writer",
    goal="Write insightful and factually accurate "
         "opinion piece about the topic: {topic}",
    backstory="You're working on a writing "
              "a new opinion piece about the topic: {topic}. "
              "You base your writing on the work of "
              "the Content Planner, who provides an outline "
              "and relevant context about the topic. "
              "You follow the main objectives and "
              "direction of the outline, "
              "as provide by the Content Planner. "
              "You also provide objective and impartial insights "
              "and back them up with information "
              "provide by the Content Planner. "
              "You acknowledge in your opinion piece "
              "when your statements are opinions "
              "as opposed to objective statements.",
    allow_delegation=False,
    verbose=True
)

### Agent: Editor

In [6]:
editor = Agent(
    role="Editor",
    goal="Edit a given blog post to align with "
         "the writing style of the organization. ",
    backstory="You are an editor who receives a blog post "
              "from the Content Writer. "
              "Your goal is to review the blog post "
              "to ensure that it follows journalistic best practices,"
              "provides balanced viewpoints "
              "when providing opinions or assertions, "
              "and also avoids major controversial topics "
              "or opinions when possible.",
    allow_delegation=False,
    verbose=True
)

## Creating Tasks

- Define your Tasks, and provide them a `description`, `expected_output` and `agent`.
Description: What you expect the task acually to be. What you expect the agent to do.

Expected_Output: Forcing function, what you want as a final outcome.


### Task: Plan

In [9]:
plan = Task(
    description=(
        "1. Prioritize the latest trends, key players, "
            "and noteworthy news on {topic}.\n"
        "2. Identify the target audience, considering "
            "their interests and pain points.\n"
        "3. Develop a detailed content outline including "
            "an introduction, key points, and a call to action.\n"
        "4. Include SEO keywords and relevant data or sources."
    ),
    expected_output="A comprehensive content plan document "
        "with an outline, audience analysis, "
        "SEO keywords, and resources.",
    agent=planner,
)

### Task: Write

In [10]:
write = Task(
    description=(
        "1. Use the content plan to craft a compelling "
            "blog post on {topic}.\n"
        "2. Incorporate SEO keywords naturally.\n"
		"3. Sections/Subtitles are properly named "
            "in an engaging manner.\n"
        "4. Ensure the post is structured with an "
            "engaging introduction, insightful body, "
            "and a summarizing conclusion.\n"
        "5. Proofread for grammatical errors and "
            "alignment with the brand's voice.\n"
    ),
    expected_output="A well-written blog post "
        "in markdown format, ready for publication, "
        "each section should have 2 or 3 paragraphs.",
    agent=writer,
)

### Task: Edit

In [11]:
edit = Task(
    description=("Proofread the given blog post for "
                 "grammatical errors and "
                 "alignment with the brand's voice."),
    expected_output="A well-written blog post in markdown format, "
                    "ready for publication, "
                    "each section should have 2 or 3 paragraphs.",
    agent=editor
)

## Creating the Crew

- Create your crew of Agents
- Pass the tasks to be performed by those agents.
    - **Note**: *For this simple example*, the tasks will be performed sequentially (i.e they are dependent on each other), so the _order_ of the task in the list _matters_.
- `verbose=2` allows you to see all the logs of the execution. 

In [12]:
crew = Crew(
    agents=[planner, writer, editor],
    tasks=[plan, write, edit],
    verbose=2
)

## Running the Crew

**Note**: LLMs can provide different outputs for they same input, so what you get might be different than what you see in the video.

In [13]:
result = crew.kickoff(inputs={"topic": "Artificial Intelligence"})

 [DEBUG]: == Working Agent: Content Planner
 [INFO]: == Starting Task: 1. Prioritize the latest trends, key players, and noteworthy news on Artificial Intelligence.
2. Identify the target audience, considering their interests and pain points.
3. Develop a detailed content outline including an introduction, key points, and a call to action.
4. Include SEO keywords and relevant data or sources.


> Entering new CrewAgentExecutor chain...
I now can give a great answer

Final Answer: 

Content Plan: 

Topic: Artificial Intelligence 

Outline:

I. Introduction
- Brief overview of Artificial Intelligence 
- Importance of AI in various industries 

II. Latest Trends in Artificial Intelligence
- Advancements in machine learning algorithms 
- Growth of AI-powered chatbots and virtual assistants 
- Integration of AI in autonomous vehicles 

III. Key Players in the Artificial Intelligence Industry
- Google's DeepMind 
- IBM Watson 
- Amazon Web Services 

IV. Noteworthy News in Artificial Intelli

I now can give a great answer

Final Answer:
# Artificial Intelligence: Revolutionizing Industries and Shaping Our Future

## Introduction

Artificial Intelligence (AI) has rapidly become a transformative force in various industries, revolutionizing the way we work and live. From enhancing customer experiences to optimizing business operations, the importance of AI cannot be overstated. This advanced technology simulates human intelligence processes, enabling machines to learn, analyze data, and make decisions with minimal human intervention.

## Latest Trends in Artificial Intelligence

The latest trends in AI showcase remarkable advancements in machine learning algorithms, leading to more accurate predictions and insights. Moreover, the growth of AI-powered chatbots and virtual assistants has streamlined communication channels and improved customer service interactions. The integration of AI in autonomous vehicles is another significant trend, promising safer and more efficient trans

- Display the results of your execution as markdown in the notebook.

In [14]:
from IPython.display import Markdown
Markdown(result)

# Artificial Intelligence: Revolutionizing Industries and Shaping Our Future

## Introduction

Artificial Intelligence (AI) has rapidly become a transformative force in various industries, revolutionizing the way we work and live. From enhancing customer experiences to optimizing business operations, the importance of AI cannot be overstated. This advanced technology simulates human intelligence processes, enabling machines to learn, analyze data, and make decisions with minimal human intervention.

## Latest Trends in Artificial Intelligence

The latest trends in AI showcase remarkable advancements in machine learning algorithms, leading to more accurate predictions and insights. Moreover, the growth of AI-powered chatbots and virtual assistants has streamlined communication channels and improved customer service interactions. The integration of AI in autonomous vehicles is another significant trend, promising safer and more efficient transportation systems.

## Key Players in the Artificial Intelligence Industry

Leading the AI industry are key players such as Google's DeepMind, known for its cutting-edge research in AI and machine learning. IBM Watson stands out for its cognitive computing capabilities, empowering businesses with data-driven insights. Amazon Web Services (AWS) offers a range of AI services, enabling organizations to leverage AI technologies for various applications.

## Noteworthy News in Artificial Intelligence

Recent breakthroughs in natural language processing have significantly improved AI's ability to understand and generate human language. However, ethical considerations in AI development remain a critical issue, raising questions about data privacy and algorithmic biases. The impact of AI on the job market is another important aspect to consider, as automation and AI technologies reshape the workforce.

## Target Audience Analysis

Our target audience includes tech-savvy individuals keen on exploring emerging technologies, business professionals seeking to implement AI solutions for strategic growth, and students/researchers delving into the complexities of artificial intelligence. By catering to these diverse groups, we aim to provide valuable insights and resources on AI developments.

## SEO Keywords

Stay updated on Artificial Intelligence trends, discover the top AI companies driving innovation, get the latest AI news updates, and explore the future of AI technology with our comprehensive blog post. 

## Call to Action

We encourage readers to stay informed on AI developments, explore AI resources, and share their thoughts or experiences with AI. By engaging with AI-related content and discussions, we can collectively shape the future of this transformative technology.

## Resources

For in-depth industry insights, refer to industry reports from Gartner and Forrester. Stay updated on AI news and trends with publications like TechCrunch and Wired. Dive into research papers from academic institutions to explore the latest advancements in artificial intelligence.

In conclusion, Artificial Intelligence continues to push boundaries and redefine possibilities across various sectors. By staying informed and actively engaging with AI advancements, we can harness the power of this technology to create a more efficient, innovative, and inclusive future.

## Try it Yourself

- Pass in a topic of your choice and see what the agents come up with!

In [15]:
topic = "Virtual cloths try on for e-commerce platform."
result = crew.kickoff(inputs={"topic": topic})

 [DEBUG]: == Working Agent: Content Planner
 [INFO]: == Starting Task: 1. Prioritize the latest trends, key players, and noteworthy news on Virtual cloths try on for e-commerce platform..
2. Identify the target audience, considering their interests and pain points.
3. Develop a detailed content outline including an introduction, key points, and a call to action.
4. Include SEO keywords and relevant data or sources.


> Entering new CrewAgentExecutor chain...
I now can give a great answer

Final Answer: 

Title: Revolutionizing E-Commerce: Virtual Clothes Try On for Seamless Shopping Experience

Introduction:
- Brief overview of virtual clothes try on technology for e-commerce platforms
- Importance of implementing this feature for enhancing user experience and increasing conversion rates

Key Points:
1. Latest Trends:
- Integration of augmented reality (AR) and artificial intelligence (AI) for accurate virtual try-ons
- Customization options for users to personalize their virtual fitti

I now can give a great answer

Final Answer:

# Revolutionizing E-Commerce: Virtual Clothes Try On for Seamless Shopping Experience

In today's digital age, virtual clothes try-on technology is revolutionizing the e-commerce industry, providing users with a seamless shopping experience from the comfort of their own homes. By integrating augmented reality (AR) and artificial intelligence (AI), e-commerce platforms are now offering accurate virtual try-ons, allowing customers to visualize how clothing items will look and fit before making a purchase. This innovative feature not only enhances the user experience but also increases conversion rates for online retailers.

**Latest Trends**

One of the latest trends in virtual clothes try-on technology is the customization options available to users. Customers can now personalize their virtual fitting experience by adjusting colors, sizes, and styles to suit their preferences. Additionally, e-commerce platforms are collaborating with fashion

In [16]:
Markdown(result)

# Revolutionizing E-Commerce: Virtual Clothes Try On for Seamless Shopping Experience

In today's digital age, virtual clothes try-on technology is revolutionizing the e-commerce industry, providing users with a seamless shopping experience from the comfort of their own homes. By integrating augmented reality (AR) and artificial intelligence (AI), e-commerce platforms are now offering accurate virtual try-ons, allowing customers to visualize how clothing items will look and fit before making a purchase. This innovative feature not only enhances the user experience but also increases conversion rates for online retailers.

**Latest Trends**

One of the latest trends in virtual clothes try-on technology is the customization options available to users. Customers can now personalize their virtual fitting experience by adjusting colors, sizes, and styles to suit their preferences. Additionally, e-commerce platforms are collaborating with fashion brands and influencers to offer exclusive virtual try-on collections, creating a unique and engaging shopping experience for users.

**Key Players**

Leading e-commerce platforms such as Amazon, ASOS, and Zara have already implemented virtual try-on features to cater to the growing demand for seamless online shopping experiences. Software companies like Zeekit, Vue.ai, and MemoMi are also providing virtual fitting solutions for retailers, enabling them to offer a more interactive and personalized shopping experience to their customers.

**Noteworthy News**

Recent success stories of e-commerce businesses that have implemented virtual try-on technology have seen a significant boost in sales, showcasing the effectiveness of this innovative feature. Advancements in AR technology continue to improve the realism of virtual fitting experiences, making it easier for customers to make informed purchasing decisions when shopping online.

By incorporating virtual clothes try-on technology, e-commerce platforms can better engage with tech-savvy Millennials and Gen Z consumers who value personalized shopping experiences. Fashion enthusiasts looking for convenient ways to try on clothes before making online purchases can now enjoy a more interactive and immersive shopping experience. E-commerce business owners can also benefit from implementing virtual fitting technology to improve customer engagement, reduce returns, and stay ahead of the competition in the ever-evolving fashion industry.

**Call to Action**

I encourage readers to explore virtual try-on options on their favorite e-commerce platforms to experience the convenience and accuracy of virtual clothes try-on technology. E-commerce businesses should consider implementing virtual fitting technology to enhance the user experience, increase conversion rates, and ultimately drive sales in the competitive online fashion market.

Overall, virtual clothes try-on technology is transforming the way we shop online, providing a more interactive and personalized experience for both consumers and businesses in the fashion industry. Embracing this innovative technology is essential for staying relevant and competitive in today's rapidly evolving e-commerce landscape.

<a name='1'></a>
 ## Other Popular Models as LLM for your Agents

#### Hugging Face (HuggingFaceHub endpoint)

```Python
from langchain_community.llms import HuggingFaceHub

llm = HuggingFaceHub(
    repo_id="HuggingFaceH4/zephyr-7b-beta",
    huggingfacehub_api_token="<HF_TOKEN_HERE>",
    task="text-generation",
)

### you will pass "llm" to your agent function
```

#### Mistral API

```Python
OPENAI_API_KEY=your-mistral-api-key
OPENAI_API_BASE=https://api.mistral.ai/v1
OPENAI_MODEL_NAME="mistral-small"
```

#### Cohere

```Python
from langchain_community.chat_models import ChatCohere
# Initialize language model
os.environ["COHERE_API_KEY"] = "your-cohere-api-key"
llm = ChatCohere()

### you will pass "llm" to your agent function
```

### For using Llama locally with Ollama and more, checkout the crewAI documentation on [Connecting to any LLM](https://docs.crewai.com/how-to/LLM-Connections/).